# Generador de canastas: ferretería La Comunidad

Genera un año de tiquetes sintéticos de una ferretería ficticia y los escribe como **Delta tables** en el Lakehouse.
Es la demo de la charla *La Comunidad del Dato* (Comunidad de Data & AI Costa Rica, 2026-11-23).

Extiende el [generador de datos sintéticos](https://github.com/JavierSQL/GeneradorDatosSinteticos) con un tipo de
dataset nuevo, `baskets`: cada tiquete nace de una **misión** (el motivo del viaje) que arrastra acompañantes con
probabilidades. El YAML es el conocimiento del mostrador escrito como datos.

## Antes de correr

1. Lakehouse adjunto **con schemas habilitados**.
2. Subir `ferreteria_canastas.yml` a `Files/config/` del Lakehouse.
3. Runtime con PyYAML (viene en los runtimes de Fabric; si falta, la celda de imports lo dice).

## Qué escribe

| Tabla | Schema | Va al modelo semántico |
|---|---|---|
| `dim_sucursal` | `Ferreteria` | Sí (en el modelo "antes", sin `Formato`) |
| `dim_articulo` | `Ferreteria` | Sí |
| `fact_venta_linea` | `Ferreteria` | Sí |
| `validacion_mision` | `Ferreteria_Validacion` | **No**: es la verdad de terreno para validar |

Todo es ficticio: empresa, sucursales, artículos y precios.

## Parámetros

In [ ]:
YAML_PATH = "/lakehouse/default/Files/config/ferreteria_canastas.yml"
TIQUETES = None                      # None = lo que diga el YAML (500 mil). Para una prueba rápida: 20000
SCHEMA = None                        # None = dataset.schema del YAML
SCHEMA_VALIDACION = "Ferreteria_Validacion"

In [ ]:
try:
    import yaml  # noqa: F401
except ImportError:
    raise ImportError("Falta PyYAML: correr  %pip install pyyaml  en una celda y reiniciar la sesión")

## Generador

Misma lógica que `generar_canastas.py`.

In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yaml


def cargar_config(path):
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def construir_catalogo(cfg, rng):
    subs, arts = [], []
    art_key = 1
    for d in cfg["catalogo"]:
        for l in d["lineas"]:
            for s in l["sublineas"]:
                sub = {
                    "Departamento": d["departamento"],
                    "Linea": l["linea"],
                    "Sublinea": s["sublinea"],
                    "Tipo": s.get("tipo", "producto"),
                    "PrecioMin": s["precio"][0],
                    "PrecioMax": s["precio"][1],
                    "CantMin": s["cantidad"][0],
                    "CantMax": s["cantidad"][1],
                    "PesoMision": s.get("peso_mision", 0.0),
                    "PesoRelleno": s.get("peso_relleno", 0.0),
                    "Sola": s.get("sola", 0.4),
                }
                subs.append(sub)
                nombres = s.get("articulos", 5)
                if isinstance(nombres, int):
                    nombres = [f"{s['sublinea'].title()} {i:02d}" for i in range(1, nombres + 1)]
                # popularidad tipo Zipf dentro de la sublínea: el primer artículo es el más vendido
                pop = 1.0 / np.arange(1, len(nombres) + 1)
                pop = pop / pop.sum()
                for i, n in enumerate(nombres):
                    precio = rng.uniform(sub["PrecioMin"], sub["PrecioMax"])
                    precio = max(50, round(precio / 50) * 50)
                    arts.append({
                        "ArticuloKey": art_key,
                        "ArticuloCodigo": f"{d['departamento'][:3]}-{art_key:05d}",
                        "ArticuloNombre": n,
                        "Departamento": sub["Departamento"],
                        "Linea": sub["Linea"],
                        "Sublinea": sub["Sublinea"],
                        "Tipo": sub["Tipo"],
                        "PrecioLista": float(precio),
                        "_pop": pop[i],
                    })
                    art_key += 1
    return pd.DataFrame(subs), pd.DataFrame(arts)


def generar(cfg, tiquetes=None):
    ds = cfg["dataset"]
    rng = np.random.default_rng(ds.get("seed", 42))
    n_tiq = tiquetes or ds["tiquetes"]
    lpf = ds.get("lineas_por_factura", 20)

    subs, arts = construir_catalogo(cfg, rng)
    sub_idx = {s: i for i, s in enumerate(subs["Sublinea"])}

    # artículos por sublínea, con su popularidad
    arts_por_sub = {s: g for s, g in arts.groupby("Sublinea")}
    art_keys = {s: g["ArticuloKey"].to_numpy() for s, g in arts_por_sub.items()}
    art_pop = {s: (g["_pop"] / g["_pop"].sum()).to_numpy() for s, g in arts_por_sub.items()}
    art_precio = dict(zip(arts["ArticuloKey"], arts["PrecioLista"]))

    # sucursales
    suc = pd.DataFrame(cfg["sucursales"])
    suc_p = (suc["peso"] / suc["peso"].sum()).to_numpy()

    # fechas
    d0 = date.fromisoformat(ds["desde"])
    d1 = date.fromisoformat(ds["hasta"])
    dias = [d0 + timedelta(days=i) for i in range((d1 - d0).days + 1)]
    pdia = np.array([ds["peso_dia_semana"][d.weekday()] for d in dias], dtype=float)
    pdia /= pdia.sum()

    # misiones y relleno
    pm = subs["PesoMision"].to_numpy(dtype=float)
    pm = pm / pm.sum()
    pr = subs["PesoRelleno"].to_numpy(dtype=float)
    pr = pr / pr.sum()
    misiones = {m["motivo"]: m for m in cfg.get("misiones", [])}
    media_rel = cfg.get("relleno", {}).get("media_lineas", 1.0)
    flete = cfg.get("flete")

    pg = cfg.get("pedidos_grandes", {})
    pg_mask = subs["Departamento"].isin(pg.get("departamentos", [])).to_numpy()
    pg_w = (subs["PesoMision"] + subs["PesoRelleno"]).to_numpy(dtype=float) * pg_mask
    pg_w = pg_w / pg_w.sum()
    pg_mult = pg.get("cantidad_multiplicador", 1)

    lineas, valid = [], []
    factura_id = 0
    proforma_id = 0

    def linea_de(sub, mult=1):
        i = sub_idx[sub]
        ak = rng.choice(art_keys[sub], p=art_pop[sub])
        cant = int(rng.integers(subs.at[i, "CantMin"], subs.at[i, "CantMax"] + 1)) * mult
        return ak, cant

    suc_idx_all = rng.choice(len(suc), size=n_tiq, p=suc_p)
    dia_idx_all = rng.choice(len(dias), size=n_tiq, p=pdia)

    for t in range(n_tiq):
        s = suc.iloc[suc_idx_all[t]]
        fecha = dias[dia_idx_all[t]]
        cred_p = s.get("credito_prob", ds.get("credito_prob", 0.0))
        if pd.isna(cred_p):
            cred_p = ds.get("credito_prob", 0.0)
        condicion = "Credito" if rng.random() < cred_p else "Contado"
        es_grande = rng.random() < pg.get("prob_por_formato", {}).get(s["formato"], 0.0)

        items = []  # (sublinea, articulo, cantidad)
        if es_grande:
            n = int(rng.integers(pg["lineas"][0], pg["lineas"][1] + 1))
            n = min(n, int(pg_mask.sum()))
            elegidas = rng.choice(len(subs), size=n, replace=False, p=pg_w)
            for i in elegidas:
                sub = subs.at[i, "Sublinea"]
                ak, cant = linea_de(sub, pg_mult)
                items.append((sub, ak, cant))
            mision, variante = "PEDIDO GRANDE", None
        else:
            mi = rng.choice(len(subs), p=pm)
            mision = subs.at[mi, "Sublinea"]
            variante = None
            presentes = {mision}
            ak, cant = linea_de(mision)
            items.append((mision, ak, cant))
            if rng.random() >= subs.at[mi, "Sola"]:
                m = misiones.get(mision, {})
                extra = []
                if "variantes" in m:
                    vs = m["variantes"]
                    v = vs[rng.choice(len(vs), p=np.array([x["prob"] for x in vs]) / sum(x["prob"] for x in vs))]
                    variante = v["nombre"]
                    extra += v.get("agrega", [])
                extra += m.get("acompanantes", [])
                for a in extra:
                    if a["sublinea"] not in presentes and rng.random() < a["prob"]:
                        presentes.add(a["sublinea"])
                        ak, cant = linea_de(a["sublinea"])
                        items.append((a["sublinea"], ak, cant))
                k = rng.poisson(media_rel)
                if k:
                    cand = rng.choice(len(subs), size=k, replace=False, p=pr)
                    for i in cand:
                        sub = subs.at[i, "Sublinea"]
                        if sub not in presentes:
                            presentes.add(sub)
                            ak, cant = linea_de(sub)
                            items.append((sub, ak, cant))

        monto_tiq = sum(art_precio[ak] * c for _, ak, c in items)
        if flete and monto_tiq >= flete["monto_minimo"] and rng.random() < flete["prob"]:
            ak, cant = linea_de(flete["sublinea"])
            items.append((flete["sublinea"], ak, cant))

        # el POS parte la compra cada `lpf` líneas; las facturas comparten proforma
        partes = [items[i:i + lpf] for i in range(0, len(items), lpf)]
        proforma = None
        if len(partes) > 1:
            proforma_id += 1
            proforma = f"PF-{proforma_id:06d}"
        for parte in partes:
            factura_id += 1
            num = f"{s['codigo']}-{factura_id:07d}"
            for sub, ak, cant in parte:
                precio = art_precio[ak]
                lineas.append((factura_id, num, proforma, fecha, s["codigo"], ak, cant, precio, cant * precio, condicion))
            valid.append((factura_id, t + 1, mision, variante, es_grande))

    fact = pd.DataFrame(lineas, columns=["FacturaID", "FacturaNumero", "Proforma", "Fecha", "SucursalCodigo",
                                         "ArticuloKey", "Cantidad", "PrecioUnitario", "Monto", "Condicion"])
    val = pd.DataFrame(valid, columns=["FacturaID", "TiqueteOrigen", "Mision", "Variante", "EsPedidoGrande"])

    # ABC del ERP: Pareto por monto anual de artículo; servicios siempre C
    abc_cfg = cfg.get("abc", {})
    m = fact.groupby("ArticuloKey")["Monto"].sum().reindex(arts["ArticuloKey"], fill_value=0)
    orden = m.sort_values(ascending=False)
    acum = orden.cumsum() / orden.sum()
    abc = pd.Series(np.where(acum <= abc_cfg.get("corte_a", 0.8), "A",
                             np.where(acum <= abc_cfg.get("corte_b", 0.95), "B", "C")), index=orden.index)
    arts["ClasificacionABC"] = arts["ArticuloKey"].map(abc)
    arts.loc[arts["Tipo"] == "servicio", "ClasificacionABC"] = abc_cfg.get("servicios", "C")

    dim_suc = suc.rename(columns={"codigo": "SucursalCodigo", "nombre": "SucursalNombre",
                                  "provincia": "Provincia", "formato": "Formato"})[
        ["SucursalCodigo", "SucursalNombre", "Provincia", "Formato"]]
    dim_art = arts.drop(columns=["_pop"])
    return {"dim_sucursal": dim_suc, "dim_articulo": dim_art,
            "fact_venta_linea": fact, "_validacion_mision": val}

## Validación

Misma lógica que `analizar_canastas.py`: confirma que los patrones plantados aparecen.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

CLAVE = ["TORNILLOS GYPSUM", "CEMENTO GRIS", "ACRILICAS", "ANTICORROSIVOS", "PISO CERAMICO",
         "ACCESORIOS PVC", "SERV CORTE", "COPIA DE LLAVES", "TABLEROS MELAMINA"]


def cargar(d):
    d = Path(d)
    return {n: pd.read_csv(d / f"{n}.csv") for n in
            ["dim_sucursal", "dim_articulo", "fact_venta_linea", "_validacion_mision"]}


def tiquetes_mostrador(t):
    f = t["fact_venta_linea"].merge(t["dim_sucursal"], on="SucursalCodigo") \
                             .merge(t["dim_articulo"][["ArticuloKey", "Sublinea", "Tipo", "ClasificacionABC"]], on="ArticuloKey")
    f = f[(f["Formato"] == "Tienda") & (f["Proforma"].isna()) & (f["Sublinea"] != "FLETE") & (f["Monto"] > 0)]
    return f


def metricas(f, val):
    ts = f.groupby(["FacturaID", "Sublinea", "Tipo"], as_index=False)["Monto"].sum()
    n_sub = ts.groupby("FacturaID")["Sublinea"].transform("count")
    tot = ts.groupby("FacturaID")["Monto"].transform("sum")
    ts["Sola"] = n_sub == 1
    rank = ts.sort_values(["FacturaID", "Monto", "Sublinea"], ascending=[True, False, True]) \
             .groupby("FacturaID").cumcount()
    ts["Dominante"] = rank.reindex(ts.index) == 0
    ts["MontoTiquete"] = tot
    # Regla del Concilio: si el tiquete trae un servicio, el servicio es el motivo
    ts["EsServicio"] = ts["Tipo"] == "servicio"
    rank2 = ts.sort_values(["FacturaID", "EsServicio", "Monto", "Sublinea"], ascending=[True, False, False, True])               .groupby("FacturaID").cumcount()
    ts["DominanteRegla"] = rank2.reindex(ts.index) == 0
    ts = ts.merge(val[["FacturaID", "Mision"]], on="FacturaID")
    ts["EsMision"] = ts["Mision"] == ts["Sublinea"]
    N = ts["FacturaID"].nunique()
    g = ts.groupby("Sublinea")
    r = pd.DataFrame({
        "Tiquetes": g.size(),
        "Pen": 100 * g.size() / N,
        "Sola": 100 * g["Sola"].mean(),
        "MotivoMonto": 100 * g["Dominante"].mean(),   # sola => dominante
        "MotivoRegla": 100 * g["DominanteRegla"].mean(),
        "MotivoReal": 100 * g["EsMision"].mean(),
        "Canasta": g["MontoTiquete"].median(),
        "MontoTotal": g["Monto"].sum(),
    })
    r["RankPen"] = r["Pen"].rank(ascending=False).astype(int)
    r["RankMonto"] = r["MontoTotal"].rank(ascending=False).astype(int)
    # perfil (cortes: penetración 1 %, motivo 50 %) con las dos definiciones
    for col, nom in [("MotivoMonto", "PerfilMonto"), ("MotivoRegla", "PerfilRegla")]:
        r[nom] = [("Imán" if m >= 50 else "Acompañante") if p >= 1 else ("Nicho" if m >= 50 else "Relleno")
                  for p, m in zip(r["Pen"], r[col])]
    return r, N, ts


def validar(t):
    """t: dict con dim_sucursal, dim_articulo, fact_venta_linea, _validacion_mision (DataFrames de pandas)."""
    f = tiquetes_mostrador(t)
    r, N, ts = metricas(f, t["_validacion_mision"])
    abc = t["dim_articulo"].groupby("Sublinea")["ClasificacionABC"].min()
    r["ABC"] = abc
    canasta_global = ts.groupby("FacturaID")["MontoTiquete"].first().median()

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    print(f"Tiquetes de mostrador: {N:,}   canasta mediana global: {canasta_global:,.0f}\n")
    print("== Top 12 por penetración (el oro común) ==")
    print(r.sort_values("Pen", ascending=False).head(12)[
        ["RankPen", "RankMonto", "Pen", "Sola", "MotivoMonto", "MotivoReal", "Canasta", "ABC"]].round(1))
    print("\n== Sublíneas clave ==")
    print(r.loc[[c for c in CLAVE if c in r.index],
                ["RankPen", "Pen", "MotivoMonto", "MotivoRegla", "MotivoReal", "Canasta", "ABC",
                 "PerfilMonto", "PerfilRegla"]].round(1))
    print("\n== Imanes clasificados C en el ERP ==")
    for col in ["PerfilMonto", "PerfilRegla"]:
        print(f"  con {col}: {list(r[(r[col] == 'Imán') & (r['ABC'] == 'C')].index)}")

    # arrastre: en tiquetes cuya misión fue X, % que llevó Y
    print("\n== Arrastre (tiquetes con canasta) ==")
    multi = ts[ts.groupby("FacturaID")["Sublinea"].transform("count") > 1]
    for a, bs in {"SERV CORTE": ["TABLEROS MELAMINA", "CANTOS", "BISAGRAS", "TORNILLOS MELAMINA"],
                  "ACRILICAS": ["BROCHAS", "FELPAS Y RODILLOS"],
                  "PISO CERAMICO": ["PEGA PISOS"],
                  "ANTICORROSIVOS": ["SOLVENTES"]}.items():
        fa = set(multi.loc[multi["Sublinea"] == a, "FacturaID"])
        base = set(multi["FacturaID"]) - fa
        for b in bs:
            fb = set(multi.loc[multi["Sublinea"] == b, "FacturaID"])
            pa = 100 * len(fa & fb) / max(1, len(fa))
            pb = 100 * len(base & fb) / max(1, len(base))
            print(f"  {a:15s} -> {b:20s} {pa:5.1f} %  (resto {pb:4.1f} %, lift {pa / max(pb, 0.01):5.1f})")

    # vista ingenua del agente "antes": todas las sucursales, factura = tiquete
    fa = t["fact_venta_linea"].merge(t["dim_articulo"][["ArticuloKey", "Sublinea"]], on="ArticuloKey")
    nv = fa["FacturaID"].nunique()
    pen_naive = (fa.groupby("Sublinea")["FacturaID"].nunique() / nv * 100).sort_values(ascending=False)
    venta_naive = fa.groupby("Sublinea")["Monto"].sum().sort_values(ascending=False)
    print("\n== Vista ingenua (agente 'antes'): top 5 por facturas y por venta ==")
    print(pd.DataFrame({"PorFacturas": pen_naive.head(5).index, "PorVenta": venta_naive.head(5).index}))
    bod = t["fact_venta_linea"].merge(t["dim_sucursal"], on="SucursalCodigo")
    print(f"\nBodega: {100 * (bod['Formato'] == 'Bodega').mean():.1f} % de las líneas, "
          f"{100 * bod.loc[bod['Formato'] == 'Bodega', 'Monto'].sum() / bod['Monto'].sum():.1f} % del monto; "
          f"facturas con proforma: {100 * bod.drop_duplicates('FacturaID')['Proforma'].notna().mean():.1f} %")

## Generar

In [ ]:
cfg = cargar_config(YAML_PATH)
tablas = generar(cfg, TIQUETES)
for nombre, df in tablas.items():
    print(f"{nombre}: {len(df):,} filas")

In [ ]:
validar(tablas)

## Escribir Delta tables

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

schema = SCHEMA or cfg["dataset"]["schema"]
for sch in {schema, SCHEMA_VALIDACION}:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {sch}")

destinos = {"dim_sucursal": schema, "dim_articulo": schema,
            "fact_venta_linea": schema, "_validacion_mision": SCHEMA_VALIDACION}

for nombre, df in tablas.items():
    df = df.copy()
    for c in df.columns:                       # columnas de texto con nulos: tipo string explícito
        if df[c].dtype == object and c != "Fecha":
            df[c] = df[c].astype("string")
    tabla = f"{destinos[nombre]}.{nombre.lstrip('_')}"
    (spark.createDataFrame(df)
          .write.mode("overwrite").option("overwriteSchema", "true")
          .format("delta").saveAsTable(tabla))
    print(f"✓ {tabla}: {spark.table(tabla).count():,} filas")

## Siguiente: modelo semántico

- **Modelo "antes"**: las tres tablas tal cual. Sin `Formato` en la sucursal, sin descripciones, sin rol de sublínea.
  `SERV CORTE` sin describir.
- **Modelo "después"**: `Formato` visible y descrito, rol de sublínea (imán, acompañante, nicho, relleno) con
  definición y owner, la regla *"si el tiquete trae un servicio, el servicio es el motivo"*, y sinónimos
  ("corte de melamina", "cortar tablero").

Detalle en `03 Demo/demo-imanes-ferreteria.md`.